In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable


In [0]:
df=spark.read.format("csv").option("inferschema","true").option("header","true")\
    .load("/Volumes/aws_catalog/default/aws_databricks/project/orders/landing/*.csv")\
    .withColumn("read_timestamp",current_timestamp())\
    .select("*","_metadata.file_name","_metadata.file_size")
display(df.limit(5))

In [0]:
print(df.count())

In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable("fmcg.bronze.orders")

In [0]:
df_orders=spark.read.table("fmcg.bronze.orders")
display(df_orders.limit(5))

In [0]:
df_orders=df_orders.filter(col("order_qty").isNotNull())
print(df_orders.count())

In [0]:
df_orders=df_orders\
    .withColumn("customer_id", when(col("customer_id").rlike("^[0-9]+$"), col("customer_id")).otherwise("999999").cast("string"))


In [0]:
df_orders=df_orders.withColumn("order_placement_date", regexp_replace(col("order_placement_date"),r"^[A-Za-z]+,\s*", ""))
display(df_orders.limit(10))

In [0]:
df_orders = df_orders.withColumn(
    "order_placement_date",
    coalesce(
        try_to_date("order_placement_date", "yyyy/MM/dd"),
        try_to_date("order_placement_date", "dd-MM-yyyy"),
        try_to_date("order_placement_date", "dd/MM/yyyy"),
        try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

In [0]:
display(df_orders.limit(5))

In [0]:
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])
df_orders=df_orders.withColumn("product_id", col("product_id").cast("string"))
df_orders.printSchema()

In [0]:
df_orders.count()

In [0]:
df_orders.agg(
    max("order_placement_date").alias("max_date"),
    min("order_placement_date").alias("min_date")
    ).show()

In [0]:
df_products=spark.table("fmcg.silver.products")
df_joined=df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"],df_products["product_code"])

display(df_joined.limit(5))

In [0]:
df_joined.write.format("delta").option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite").saveAsTable("fmcg.silver.orders")

In [0]:
delta_table=DeltaTable.forName(spark, "fmcg.silver.orders")
delta_table.alias("silver").merge(df_joined.alias("bronze"), 
                                  "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM fmcg.silver.orders")

display(df_gold.limit(2))

In [0]:
df_gold.count()

In [0]:
df_gold.write.format("delta").option("mergeschema","true")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite").saveAsTable("fmcg.gold.sb_fact_orders")

In [0]:
gold_delta = DeltaTable.forName(spark, "fmcg.gold.sb_fact_orders")
gold_delta.alias("source").merge(
    df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM fmcg.gold.sb_fact_orders")
df_child.show(10)

In [0]:
df_child.count()

In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

In [0]:
gold_parent_delta = DeltaTable.forName(spark, "fmcg.gold.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
files=dbutils.fs.ls("/Volumes/aws_catalog/default/aws_databricks/project/orders/landing/")
for file in files:
    dbutils.fs.mv(file.path,f"/Volumes/aws_catalog/default/aws_databricks/project/orders/processed/{file.name}",True)